# FETAL-CHD: 44-model Kaggle benchmark

This notebook loads the CARDIUM dataset and a **real 44-row model manifest**, downloads and caches each checkpoint, evaluates all available models on the same test split, and packages the results and weights into a ZIP file.

> The manifest must contain the exact checkpoint URLs and licenses. Do not replace unavailable checkpoints with invented URLs. Failed/private models are recorded as failed and are not counted as valid scores.

In [ ]:
# Kaggle Settings: enable Internet and GPU if the selected models require them.
# The code below discovers the real mounted paths, including Kaggle's
# generated dataset names and extra nested folders.

from pathlib import Path
import torch

input_root = Path("/kaggle/input")
manifest_candidates = sorted(input_root.rglob("models.csv"))
if len(manifest_candidates) != 1:
    raise RuntimeError("models.csv was not found exactly once under /kaggle/input")
MANIFEST = str(manifest_candidates[0])

test_candidates = sorted(input_root.rglob("fold_1/test/CHD"))
if len(test_candidates) != 1:
    raise RuntimeError("CARDIUM fold_1/test/CHD was not found exactly once")
DATA_ROOT = str(test_candidates[0].parents[4])
EXPECTED_MODELS = 44
STRICT_44 = False  # Set True only after your CSV contains 44 real checkpoints.
FOLD = 1
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = "/kaggle/working/model_cache"
RESULTS_CSV = "/kaggle/working/benchmark_results.csv"
BUNDLE_ZIP = "/kaggle/working/fetal_chd_models_and_results.zip"
print({"device": DEVICE, "data": DATA_ROOT, "manifest": MANIFEST})

In [ ]:
!pip install -q timm scikit-learn tqdm pillow torchvision

In [ ]:
# Create the required project files locally; no GitHub clone or code dataset is needed.
import base64, os, sys
REPO_DIR = "/kaggle/working/FETAL-CHD"
os.makedirs(REPO_DIR, exist_ok=True)
embedded_code = {'data.py': 'IiIiRGF0YSBsb2FkaW5nIHV0aWxpdGllcyBmb3IgQ0FSRElVTSBkYXRhc2V0LiIiIg0KaW1wb3J0IG9zDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBUdXBsZSwgT3B0aW9uYWwNCmltcG9ydCB0b3JjaA0KZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0LCBEYXRhTG9hZGVyDQpmcm9tIHRvcmNodmlzaW9uIGltcG9ydCB0cmFuc2Zvcm1zDQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UNCmltcG9ydCBudW1weSBhcyBucA0KDQoNCmNsYXNzIENBUkRJVU1EYXRhc2V0KERhdGFzZXQpOg0KICAgICIiIlB5VG9yY2ggZGF0YXNldCBmb3IgQ0FSRElVTSBmZXRhbCB1bHRyYXNvdW5kIGltYWdlcy4iIiINCiAgICANCiAgICBkZWYgX19pbml0X18oDQogICAgICAgIHNlbGYsDQogICAgICAgIGRhdGFfcm9vdDogc3RyLA0KICAgICAgICBzcGxpdDogc3RyID0gJ3RyYWluJywNCiAgICAgICAgZm9sZDogaW50ID0gMSwNCiAgICAgICAgdHJhbnNmb3JtOiBPcHRpb25hbFt0cmFuc2Zvcm1zLkNvbXBvc2VdID0gTm9uZSwNCiAgICAgICAgaW5jbHVkZV9hbGw6IGJvb2wgPSBGYWxzZQ0KICAgICk6DQogICAgICAgICIiIg0KICAgICAgICBBcmdzOg0KICAgICAgICAgICAgZGF0YV9yb290OiBQYXRoIHRvIENBUkRJVU0gZGF0YXNldCByb290DQogICAgICAgICAgICBzcGxpdDogJ3RyYWluJyBvciAndGVzdCcNCiAgICAgICAgICAgIGZvbGQ6IENyb3NzLXZhbGlkYXRpb24gZm9sZCAoMSwgMiwgb3IgMykNCiAgICAgICAgICAgIHRyYW5zZm9ybTogSW1hZ2UgdHJhbnNmb3JtYXRpb25zDQogICAgICAgICAgICBpbmNsdWRlX2FsbDogSWYgVHJ1ZSwgaWdub3JlIGZvbGQgYW5kIGxvYWQgZW50aXJlIGRhdGFzZXQNCiAgICAgICAgIiIiDQogICAgICAgIHNlbGYuZGF0YV9yb290ID0gUGF0aChkYXRhX3Jvb3QpDQogICAgICAgIHNlbGYuc3BsaXQgPSBzcGxpdA0KICAgICAgICBzZWxmLmZvbGQgPSBmb2xkDQogICAgICAgIHNlbGYuaW5jbHVkZV9hbGwgPSBpbmNsdWRlX2FsbA0KICAgICAgICANCiAgICAgICAgaWYgdHJhbnNmb3JtIGlzIE5vbmU6DQogICAgICAgICAgICBzZWxmLnRyYW5zZm9ybSA9IHRyYW5zZm9ybXMuQ29tcG9zZShbDQogICAgICAgICAgICAgICAgdHJhbnNmb3Jtcy5SZXNpemUoKDIyNCwgMjI0KSksDQogICAgICAgICAgICAgICAgdHJhbnNmb3Jtcy5Ub1RlbnNvcigpLA0KICAgICAgICAgICAgICAgIHRyYW5zZm9ybXMuTm9ybWFsaXplKA0KICAgICAgICAgICAgICAgICAgICBtZWFuPVswLjQ4NSwgMC40NTYsIDAuNDA2XSwNCiAgICAgICAgICAgICAgICAgICAgc3RkPVswLjIyOSwgMC4yMjQsIDAuMjI1XQ0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgICAgIF0pDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBzZWxmLnRyYW5zZm9ybSA9IHRyYW5zZm9ybQ0KICAgICAgICANCiAgICAgICAgc2VsZi5pbWFnZXMgPSBbXQ0KICAgICAgICBzZWxmLmxhYmVscyA9IFtdDQogICAgICAgIHNlbGYuX2xvYWRfZGF0YXNldCgpDQogICAgDQogICAgZGVmIF9sb2FkX2RhdGFzZXQoc2VsZik6DQogICAgICAgICIiIkxvYWQgaW1hZ2UgcGF0aHMgYW5kIGxhYmVscyBmcm9tIGRhdGFzZXQgc3RydWN0dXJlLiIiIg0KICAgICAgICBpZiBzZWxmLmluY2x1ZGVfYWxsOg0KICAgICAgICAgICAgIyBMb2FkIGVudGlyZSBDQVJESVVNX2RhdGFzZXQgKG9yZ2FuaXplZCBhcyBDSEQgLyBOb25fQ0hEKQ0KICAgICAgICAgICAgY2FyZGl1bV9yb290ID0gc2VsZi5kYXRhX3Jvb3QgLyAnQ0FSRElVTV9kYXRhc2V0Jw0KICAgICAgICAgICAgZm9yIGxhYmVsLCBjbGFzc19uYW1lIGluIGVudW1lcmF0ZShbJ05vbl9DSEQnLCAnQ0hEJ10pOg0KICAgICAgICAgICAgICAgIGNsYXNzX2RpciA9IGNhcmRpdW1fcm9vdCAvIGNsYXNzX25hbWUNCiAgICAgICAgICAgICAgICBpZiBjbGFzc19kaXIuZXhpc3RzKCk6DQogICAgICAgICAgICAgICAgICAgIGZvciBzdWJkaXIgaW4gc29ydGVkKGNsYXNzX2Rpci5pdGVyZGlyKCkpOg0KICAgICAgICAgICAgICAgICAgICAgICAgaWYgc3ViZGlyLmlzX2RpcigpOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpbWdfZmlsZSBpbiBzb3J0ZWQoc3ViZGlyLmdsb2IoJyoucG5nJykpOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmltYWdlcy5hcHBlbmQoc3RyKGltZ19maWxlKSkNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5sYWJlbHMuYXBwZW5kKGxhYmVsKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgIyBMb2FkIGZyb20gZm9sZCBzdHJ1Y3R1cmUgKGNhcmRpdW1faW1hZ2VzKQ0KICAgICAgICAgICAgZm9sZF9yb290ID0gc2VsZi5kYXRhX3Jvb3QgLyAnY2FyZGl1bV9pbWFnZXMnIC8gJ2NhcmRpdW1faW1hZ2VzJyAvIGYnZm9sZF97c2VsZi5mb2xkfScgLyBzZWxmLnNwbGl0DQogICAgICAgICAgICANCiAgICAgICAgICAgIGZvciBsYWJlbCwgY2xhc3NfbmFtZSBpbiBlbnVtZXJhdGUoWydOb25fQ0hEJywgJ0NIRCddKToNCiAgICAgICAgICAgICAgICBjbGFzc19kaXIgPSBmb2xkX3Jvb3QgLyBjbGFzc19uYW1lDQogICAgICAgICAgICAgICAgaWYgY2xhc3NfZGlyLmV4aXN0cygpOg0KICAgICAgICAgICAgICAgICAgICBmb3IgaW1nX2ZpbGUgaW4gc29ydGVkKGNsYXNzX2Rpci5nbG9iKCcqLnBuZycpKToNCiAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYuaW1hZ2VzLmFwcGVuZChzdHIoaW1nX2ZpbGUpKQ0KICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5sYWJlbHMuYXBwZW5kKGxhYmVsKQ0KICAgIA0KICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGxlbihzZWxmLmltYWdlcykNCiAgICANCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaWR4OiBpbnQpIC0+IFR1cGxlW3RvcmNoLlRlbnNvciwgaW50LCBzdHJdOg0KICAgICAgICAiIiJSZXR1cm4gKGltYWdlX3RlbnNvciwgbGFiZWwsIGltYWdlX3BhdGgpLiIiIg0KICAgICAgICBpbWdfcGF0aCA9IHNlbGYuaW1hZ2VzW2lkeF0NCiAgICAgICAgbGFiZWwgPSBzZWxmLmxhYmVsc1tpZHhdDQogICAgICAgIA0KICAgICAgICB0cnk6DQogICAgICAgICAgICBpbWFnZSA9IEltYWdlLm9wZW4oaW1nX3BhdGgpLmNvbnZlcnQoJ1JHQicpDQogICAgICAgICAgICBpbWFnZSA9IHNlbGYudHJhbnNmb3JtKGltYWdlKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgICAgICBwcmludChmIkVycm9yIGxvYWRpbmcge2ltZ19wYXRofToge2V9IikNCiAgICAgICAgICAgICMgUmV0dXJuIGJsYWNrIGltYWdlIG9uIGVycm9yDQogICAgICAgICAgICBpbWFnZSA9IHRvcmNoLnplcm9zKDMsIDIyNCwgMjI0KQ0KICAgICAgICANCiAgICAgICAgcmV0dXJuIGltYWdlLCBsYWJlbCwgaW1nX3BhdGgNCg0KDQpjbGFzcyBDQVJESVVNRGF0YU1vZHVsZToNCiAgICAiIiJDb252ZW5pZW50IGRhdGEgbG9hZGluZyBmb3IgdHJhaW5pbmcvZXZhbHVhdGlvbi4iIiINCiAgICANCiAgICBkZWYgX19pbml0X18oDQogICAgICAgIHNlbGYsDQogICAgICAgIGRhdGFfcm9vdDogc3RyLA0KICAgICAgICBmb2xkOiBpbnQgPSAxLA0KICAgICAgICBiYXRjaF9zaXplOiBpbnQgPSAzMiwNCiAgICAgICAgbnVtX3dvcmtlcnM6IGludCA9IDAsDQogICAgICAgIGF1Z21lbnRfdHJhaW46IGJvb2wgPSBUcnVlDQogICAgKToNCiAgICAgICAgIiIiSW5pdGlhbGl6ZSBkYXRhIG1vZHVsZS4NCiAgICAgICAgDQogICAgICAgIEFyZ3M6DQogICAgICAgICAgICBkYXRhX3Jvb3Q6IFJvb3QgcGF0aCB0byBkYXRhc2V0DQogICAgICAgICAgICBmb2xkOiBDViBmb2xkICgxLCAyLCAzKQ0KICAgICAgICAgICAgYmF0Y2hfc2l6ZTogQmF0Y2ggc2l6ZSBmb3IgbG9hZGVyDQogICAgICAgICAgICBudW1fd29ya2VyczogRGF0YUxvYWRlciB3b3JrZXJzDQogICAgICAgICAgICBhdWdtZW50X3RyYWluOiBBcHBseSBhdWdtZW50YXRpb24gdG8gdHJhaW5pbmcgc2V0DQogICAgICAgICIiIg0KICAgICAgICBzZWxmLmRhdGFfcm9vdCA9IGRhdGFfcm9vdA0KICAgICAgICBzZWxmLmZvbGQgPSBmb2xkDQogICAgICAgIHNlbGYuYmF0Y2hfc2l6ZSA9IGJhdGNoX3NpemUNCiAgICAgICAgc2VsZi5udW1fd29ya2VycyA9IG51bV93b3JrZXJzDQogICAgICAgIA0KICAgICAgICB0cmFpbl90cmFuc2Zvcm0gPSB0cmFuc2Zvcm1zLkNvbXBvc2UoWw0KICAgICAgICAgICAgdHJhbnNmb3Jtcy5SZXNpemUoKDIyNCwgMjI0KSksDQogICAgICAgICAgICB0cmFuc2Zvcm1zLlJhbmRvbUhvcml6b250YWxGbGlwKDAuNSkgaWYgYXVnbWVudF90cmFpbiBlbHNlIHRyYW5zZm9ybXMuTGFtYmRhKGxhbWJkYSB4OiB4KSwNCiAgICAgICAgICAgIHRyYW5zZm9ybXMuVG9UZW5zb3IoKSwNCiAgICAgICAgICAgIHRyYW5zZm9ybXMuTm9ybWFsaXplKA0KICAgICAgICAgICAgICAgIG1lYW49WzAuNDg1LCAwLjQ1NiwgMC40MDZdLA0KICAgICAgICAgICAgICAgIHN0ZD1bMC4yMjksIDAuMjI0LCAwLjIyNV0NCiAgICAgICAgICAgICkNCiAgICAgICAgXSkgaWYgYXVnbWVudF90cmFpbiBlbHNlIE5vbmUNCiAgICAgICAgDQogICAgICAgIHNlbGYudHJhaW5fZGF0YXNldCA9IENBUkRJVU1EYXRhc2V0KA0KICAgICAgICAgICAgZGF0YV9yb290LA0KICAgICAgICAgICAgc3BsaXQ9J3RyYWluJywNCiAgICAgICAgICAgIGZvbGQ9Zm9sZCwNCiAgICAgICAgICAgIHRyYW5zZm9ybT10cmFpbl90cmFuc2Zvcm0NCiAgICAgICAgKQ0KICAgICAgICANCiAgICAgICAgc2VsZi52YWxfZGF0YXNldCA9IENBUkRJVU1EYXRhc2V0KA0KICAgICAgICAgICAgZGF0YV9yb290LA0KICAgICAgICAgICAgc3BsaXQ9J3Rlc3QnLA0KICAgICAgICAgICAgZm9sZD1mb2xkLA0KICAgICAgICApDQogICAgDQogICAgZGVmIHRyYWluX2xvYWRlcihzZWxmKSAtPiBEYXRhTG9hZGVyOg0KICAgICAgICAiIiJSZXR1cm4gdHJhaW5pbmcgRGF0YUxvYWRlci4iIiINCiAgICAgICAgcmV0dXJuIERhdGFMb2FkZXIoDQogICAgICAgICAgICBzZWxmLnRyYWluX2RhdGFzZXQsDQogICAgICAgICAgICBiYXRjaF9zaXplPXNlbGYuYmF0Y2hfc2l6ZSwNCiAgICAgICAgICAgIHNodWZmbGU9VHJ1ZSwNCiAgICAgICAgICAgIG51bV93b3JrZXJzPXNlbGYubnVtX3dvcmtlcnMsDQogICAgICAgICAgICBwaW5fbWVtb3J5PVRydWUNCiAgICAgICAgKQ0KICAgIA0KICAgIGRlZiB2YWxfbG9hZGVyKHNlbGYpIC0+IERhdGFMb2FkZXI6DQogICAgICAgICIiIlJldHVybiB2YWxpZGF0aW9uIERhdGFMb2FkZXIuIiIiDQogICAgICAgIHJldHVybiBEYXRhTG9hZGVyKA0KICAgICAgICAgICAgc2VsZi52YWxfZGF0YXNldCwNCiAgICAgICAgICAgIGJhdGNoX3NpemU9c2VsZi5iYXRjaF9zaXplLA0KICAgICAgICAgICAgc2h1ZmZsZT1GYWxzZSwNCiAgICAgICAgICAgIG51bV93b3JrZXJzPXNlbGYubnVtX3dvcmtlcnMsDQogICAgICAgICAgICBwaW5fbWVtb3J5PVRydWUNCiAgICAgICAgKQ0KICAgIA0KICAgIGRlZiBnZXRfY2xhc3NfY291bnRzKHNlbGYpIC0+IGRpY3Q6DQogICAgICAgICIiIlJldHVybiBjbGFzcyBkaXN0cmlidXRpb24uIiIiDQogICAgICAgIHRyYWluX2xhYmVscyA9IG5wLmFycmF5KHNlbGYudHJhaW5fZGF0YXNldC5sYWJlbHMpIGlmIGxlbihzZWxmLnRyYWluX2RhdGFzZXQubGFiZWxzKSA+IDAgZWxzZSBucC5hcnJheShbXSkNCiAgICAgICAgdmFsX2xhYmVscyA9IG5wLmFycmF5KHNlbGYudmFsX2RhdGFzZXQubGFiZWxzKSBpZiBsZW4oc2VsZi52YWxfZGF0YXNldC5sYWJlbHMpID4gMCBlbHNlIG5wLmFycmF5KFtdKQ0KICAgICAgICANCiAgICAgICAgaWYgbGVuKHRyYWluX2xhYmVscykgPiAwOg0KICAgICAgICAgICAgdHJhaW5fY291bnRzID0gbnAuYmluY291bnQodHJhaW5fbGFiZWxzKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgdHJhaW5fY291bnRzID0gbnAuYXJyYXkoWzAsIDBdKQ0KICAgICAgICANCiAgICAgICAgaWYgbGVuKHZhbF9sYWJlbHMpID4gMDoNCiAgICAgICAgICAgIHZhbF9jb3VudHMgPSBucC5iaW5jb3VudCh2YWxfbGFiZWxzKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgdmFsX2NvdW50cyA9IG5wLmFycmF5KFswLCAwXSkNCiAgICAgICAgDQogICAgICAgIHJldHVybiB7DQogICAgICAgICAgICAndHJhaW4nOiB7J05vbl9DSEQnOiBpbnQodHJhaW5fY291bnRzWzBdKSBpZiBsZW4odHJhaW5fY291bnRzKSA+IDAgZWxzZSAwLCANCiAgICAgICAgICAgICAgICAgICAgICAnQ0hEJzogaW50KHRyYWluX2NvdW50c1sxXSkgaWYgbGVuKHRyYWluX2NvdW50cykgPiAxIGVsc2UgMH0sDQogICAgICAgICAgICAndmFsJzogeydOb25fQ0hEJzogaW50KHZhbF9jb3VudHNbMF0pIGlmIGxlbih2YWxfY291bnRzKSA+IDAgZWxzZSAwLCANCiAgICAgICAgICAgICAgICAgICAgJ0NIRCc6IGludCh2YWxfY291bnRzWzFdKSBpZiBsZW4odmFsX2NvdW50cykgPiAxIGVsc2UgMH0NCiAgICAgICAgfQ0K', 'kaggle_benchmark.py': 'IiIiT25lLWNvbW1hbmQgYmVuY2htYXJrIHJ1bm5lciBmb3IgcHVibGlzaGVkIGZldGFsLXVsdHJhc291bmQgbW9kZWxzLg0KDQpUaGUgbW9kZWwgbWFuaWZlc3QgaXMgaW50ZW50aW9uYWxseSBleHRlcm5hbDogcHVibGlzaGVkIGNoZWNrcG9pbnRzIHVzZQ0KZGlmZmVyZW50IGFyY2hpdGVjdHVyZXMsIGxpY2Vuc2VzLCBhbmQgZG93bmxvYWQgbG9jYXRpb25zLiBUaGlzIHJ1bm5lciBuZXZlcg0KcHJldGVuZHMgdGhhdCBhbiB1bmF2YWlsYWJsZSBjaGVja3BvaW50IGlzIGEgdmFsaWQgbW9kZWwuDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBjc3YNCmltcG9ydCBoYXNobGliDQppbXBvcnQganNvbg0KaW1wb3J0IG9zDQppbXBvcnQgc2h1dGlsDQppbXBvcnQgc3VicHJvY2Vzcw0KaW1wb3J0IHVybGxpYi5yZXF1ZXN0DQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBJdGVyYWJsZQ0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCB0b3JjaA0KZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0ICgNCiAgICBhY2N1cmFjeV9zY29yZSwNCiAgICBjb25mdXNpb25fbWF0cml4LA0KICAgIGYxX3Njb3JlLA0KICAgIHByZWNpc2lvbl9zY29yZSwNCiAgICByZWNhbGxfc2NvcmUsDQogICAgcm9jX2F1Y19zY29yZSwNCikNCmZyb20gdG9yY2ggaW1wb3J0IG5uDQpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXINCmZyb20gdG9yY2h2aXNpb24gaW1wb3J0IG1vZGVscw0KDQpmcm9tIGRhdGEgaW1wb3J0IENBUkRJVU1EYXRhc2V0DQoNCg0KZGVmIGRvd25sb2FkKHVybDogc3RyLCBkZXN0aW5hdGlvbjogUGF0aCwgc2hhMjU2OiBzdHIgPSAiIikgLT4gUGF0aDoNCiAgICBkZXN0aW5hdGlvbi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIGlmIG5vdCBkZXN0aW5hdGlvbi5leGlzdHMoKToNCiAgICAgICAgdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUodXJsLCBkZXN0aW5hdGlvbikNCiAgICBpZiBzaGEyNTY6DQogICAgICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KGRlc3RpbmF0aW9uLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkNCiAgICAgICAgaWYgZGlnZXN0Lmxvd2VyKCkgIT0gc2hhMjU2Lmxvd2VyKCk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU0hBLTI1NiBtaXNtYXRjaCBmb3Ige2Rlc3RpbmF0aW9uLm5hbWV9IikNCiAgICByZXR1cm4gZGVzdGluYXRpb24NCg0KDQpkZWYgbG9hZF9tb2RlbChyb3c6IERpY3Rbc3RyLCBzdHJdLCBjYWNoZV9kaXI6IFBhdGgsIGRldmljZTogdG9yY2guZGV2aWNlKSAtPiBubi5Nb2R1bGU6DQogICAgbmFtZSA9IHJvd1sibmFtZSJdLnN0cmlwKCkNCiAgICBsb2FkZXIgPSByb3dbImxvYWRlciJdLnN0cmlwKCkubG93ZXIoKQ0KICAgIGNoZWNrcG9pbnQgPSBkb3dubG9hZChyb3dbImNoZWNrcG9pbnRfdXJsIl0sIGNhY2hlX2RpciAvIHJvd1siZmlsZW5hbWUiXSwgcm93LmdldCgic2hhMjU2IiwgIiIpKQ0KDQogICAgaWYgbG9hZGVyID09ICJ0b3JjaHNjcmlwdCI6DQogICAgICAgIG1vZGVsID0gdG9yY2guaml0LmxvYWQoc3RyKGNoZWNrcG9pbnQpLCBtYXBfbG9jYXRpb249ZGV2aWNlKQ0KICAgIGVsaWYgbG9hZGVyID09ICJ0b3JjaHZpc2lvbiI6DQogICAgICAgIGNvbnN0cnVjdG9yID0gZ2V0YXR0cihtb2RlbHMsIHJvd1siYXJjaGl0ZWN0dXJlIl0pDQogICAgICAgIG1vZGVsID0gY29uc3RydWN0b3Iod2VpZ2h0cz1Ob25lLCBudW1fY2xhc3Nlcz0yKQ0KICAgICAgICBzdGF0ZSA9IHRvcmNoLmxvYWQoY2hlY2twb2ludCwgbWFwX2xvY2F0aW9uPWRldmljZSkNCiAgICAgICAgc3RhdGUgPSBzdGF0ZS5nZXQoInN0YXRlX2RpY3QiLCBzdGF0ZSkNCiAgICAgICAgc3RhdGUgPSB7a2V5LnJlbW92ZXByZWZpeCgibW9kdWxlLiIpOiB2YWx1ZSBmb3Iga2V5LCB2YWx1ZSBpbiBzdGF0ZS5pdGVtcygpfQ0KICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGUsIHN0cmljdD1GYWxzZSkNCiAgICBlbHNlOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgZiJ7bmFtZX06IHVuc3VwcG9ydGVkIGxvYWRlciB7bG9hZGVyIXJ9OyB1c2UgdG9yY2hzY3JpcHQgb3IgdG9yY2h2aXNpb24iDQogICAgICAgICkNCiAgICByZXR1cm4gbW9kZWwudG8oZGV2aWNlKS5ldmFsKCkNCg0KDQpkZWYgcG9zaXRpdmVfcHJvYmFiaWxpdHkob3V0cHV0OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoNCiAgICBpZiBpc2luc3RhbmNlKG91dHB1dCwgKHR1cGxlLCBsaXN0KSk6DQogICAgICAgIG91dHB1dCA9IG91dHB1dFswXQ0KICAgIGlmIG91dHB1dC5uZGltID09IDEgb3Igb3V0cHV0LnNoYXBlWy0xXSA9PSAxOg0KICAgICAgICByZXR1cm4gdG9yY2guc2lnbW9pZChvdXRwdXQucmVzaGFwZSgtMSkpDQogICAgcmV0dXJuIHRvcmNoLnNvZnRtYXgob3V0cHV0LCBkaW09LTEpWzosIDFdDQoNCg0KQHRvcmNoLmluZmVyZW5jZV9tb2RlKCkNCmRlZiBldmFsdWF0ZShtb2RlbDogbm4uTW9kdWxlLCBsb2FkZXI6IERhdGFMb2FkZXIsIGRldmljZTogdG9yY2guZGV2aWNlKSAtPiBEaWN0W3N0ciwgZmxvYXRdOg0KICAgIHByb2JhYmlsaXRpZXMsIGxhYmVscyA9IFtdLCBbXQ0KICAgIGZvciBpbWFnZXMsIGJhdGNoX2xhYmVscywgXyBpbiBsb2FkZXI6DQogICAgICAgIHByb2JhYmlsaXRpZXMuZXh0ZW5kKHBvc2l0aXZlX3Byb2JhYmlsaXR5KG1vZGVsKGltYWdlcy50byhkZXZpY2UpKSkuY3B1KCkubnVtcHkoKSkNCiAgICAgICAgbGFiZWxzLmV4dGVuZChiYXRjaF9sYWJlbHMubnVtcHkoKSkNCiAgICBpZiBub3QgbGFiZWxzOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaGUgZXZhbHVhdGlvbiBzcGxpdCBpcyBlbXB0eSIpDQogICAgbGFiZWxzID0gbnAuYXNhcnJheShsYWJlbHMpDQogICAgcHJvYmFiaWxpdGllcyA9IG5wLmFzYXJyYXkocHJvYmFiaWxpdGllcykNCiAgICBwcmVkaWN0aW9ucyA9IChwcm9iYWJpbGl0aWVzID49IDAuNSkuYXN0eXBlKG5wLmludDY0KQ0KICAgIHRuLCBmcCwgZm4sIHRwID0gY29uZnVzaW9uX21hdHJpeChsYWJlbHMsIHByZWRpY3Rpb25zLCBsYWJlbHM9WzAsIDFdKS5yYXZlbCgpDQogICAgcmV0dXJuIHsNCiAgICAgICAgIm4iOiBpbnQobGFiZWxzLnNpemUpLA0KICAgICAgICAiYWNjdXJhY3kiOiBmbG9hdChhY2N1cmFjeV9zY29yZShsYWJlbHMsIHByZWRpY3Rpb25zKSksDQogICAgICAgICJwcmVjaXNpb24iOiBmbG9hdChwcmVjaXNpb25fc2NvcmUobGFiZWxzLCBwcmVkaWN0aW9ucywgemVyb19kaXZpc2lvbj0wKSksDQogICAgICAgICJyZWNhbGxfc2Vuc2l0aXZpdHkiOiBmbG9hdChyZWNhbGxfc2NvcmUobGFiZWxzLCBwcmVkaWN0aW9ucywgemVyb19kaXZpc2lvbj0wKSksDQogICAgICAgICJzcGVjaWZpY2l0eSI6IGZsb2F0KHRuIC8gKHRuICsgZnApKSBpZiB0biArIGZwIGVsc2UgMC4wLA0KICAgICAgICAiZjEiOiBmbG9hdChmMV9zY29yZShsYWJlbHMsIHByZWRpY3Rpb25zLCB6ZXJvX2RpdmlzaW9uPTApKSwNCiAgICAgICAgInJvY19hdWMiOiBmbG9hdChyb2NfYXVjX3Njb3JlKGxhYmVscywgcHJvYmFiaWxpdGllcykpDQogICAgICAgIGlmIGxlbihucC51bmlxdWUobGFiZWxzKSkgPT0gMg0KICAgICAgICBlbHNlIGZsb2F0KCJuYW4iKSwNCiAgICB9DQoNCg0KZGVmIHJlYWRfbWFuaWZlc3QocGF0aDogUGF0aCwgZXhwZWN0ZWQ6IGludCkgLT4gSXRlcmFibGVbRGljdFtzdHIsIHN0cl1dOg0KICAgIHdpdGggcGF0aC5vcGVuKG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToNCiAgICAgICAgcm93cyA9IGxpc3QoY3N2LkRpY3RSZWFkZXIoaGFuZGxlKSkNCiAgICByZXF1aXJlZCA9IHsibmFtZSIsICJsb2FkZXIiLCAiYXJjaGl0ZWN0dXJlIiwgImNoZWNrcG9pbnRfdXJsIiwgImZpbGVuYW1lIn0NCiAgICBtaXNzaW5nID0gcmVxdWlyZWQgLSBzZXQocm93c1swXSkgaWYgcm93cyBlbHNlIHJlcXVpcmVkDQogICAgaWYgbWlzc2luZzoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1hbmlmZXN0IGlzIG1pc3NpbmcgY29sdW1uczoge3NvcnRlZChtaXNzaW5nKX0iKQ0KICAgIGlmIGV4cGVjdGVkIGFuZCBsZW4ocm93cykgIT0gZXhwZWN0ZWQ6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCB7ZXhwZWN0ZWR9IG1hbmlmZXN0IHJvd3MsIGZvdW5kIHtsZW4ocm93cyl9IikNCiAgICByZXR1cm4gcm93cw0KDQoNCmRlZiBtYWluKCkgLT4gTm9uZToNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJDQVJESVVNIHJvb3QgZGlyZWN0b3J5IikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1hbmlmZXN0IiwgcmVxdWlyZWQ9VHJ1ZSwgdHlwZT1QYXRoKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0iYmVuY2htYXJrX3Jlc3VsdHMuY3N2IiwgdHlwZT1QYXRoKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZm9sZCIsIHR5cGU9aW50LCBkZWZhdWx0PTEpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MzIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1leHBlY3RlZC1tb2RlbHMiLCB0eXBlPWludCwgZGVmYXVsdD00NCkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNhY2hlIiwgZGVmYXVsdD0iL2thZ2dsZS93b3JraW5nL21vZGVsX2NhY2hlIiwgdHlwZT1QYXRoKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD0iY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQ0KICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpDQoNCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoYXJncy5kZXZpY2UpDQogICAgZGF0YXNldCA9IENBUkRJVU1EYXRhc2V0KGFyZ3MuZGF0YV9yb290LCBzcGxpdD0idGVzdCIsIGZvbGQ9YXJncy5mb2xkKQ0KICAgIGlmIG5vdCBsZW4oZGF0YXNldCk6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoDQogICAgICAgICAgICAiTm8gZm9sZCBpbWFnZXMgZm91bmQuIENvbnZlcnQgdGhlIENBUkRJVU0gZGF0YSB0byAiDQogICAgICAgICAgICAiY2FyZGl1bV9pbWFnZXMvY2FyZGl1bV9pbWFnZXMvZm9sZF9OL3Rlc3Qve0NIRCxOb25fQ0hEfS4iDQogICAgICAgICkNCiAgICBsb2FkZXIgPSBEYXRhTG9hZGVyKGRhdGFzZXQsIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlLCBudW1fd29ya2Vycz0yKQ0KDQogICAgcmVzdWx0cyA9IFtdDQogICAgZm9yIHJvdyBpbiByZWFkX21hbmlmZXN0KGFyZ3MubWFuaWZlc3QsIGFyZ3MuZXhwZWN0ZWRfbW9kZWxzKToNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgbWV0cmljcyA9IGV2YWx1YXRlKGxvYWRfbW9kZWwocm93LCBhcmdzLmNhY2hlLCBkZXZpY2UpLCBsb2FkZXIsIGRldmljZSkNCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHsibW9kZWwiOiByb3dbIm5hbWUiXSwgInN0YXR1cyI6ICJvayIsICoqbWV0cmljc30pDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXJyb3I6DQogICAgICAgICAgICByZXN1bHRzLmFwcGVuZCh7Im1vZGVsIjogcm93WyJuYW1lIl0sICJzdGF0dXMiOiAiZmFpbGVkIiwgImVycm9yIjogc3RyKGVycm9yKX0pDQogICAgYXJncy5vdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB3aXRoIGFyZ3Mub3V0Lm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6DQogICAgICAgIGtleXMgPSBzb3J0ZWQoe2tleSBmb3Igcm93IGluIHJlc3VsdHMgZm9yIGtleSBpbiByb3d9KQ0KICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihoYW5kbGUsIGZpZWxkbmFtZXM9a2V5cykNCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkNCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhyZXN1bHRzKQ0KICAgIHByaW50KGpzb24uZHVtcHMoeyJyZXN1bHRzIjogc3RyKGFyZ3Mub3V0KSwgIm1vZGVscyI6IGxlbihyZXN1bHRzKX0sIGluZGVudD0yKSkNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0K'}
for filename, encoded in embedded_code.items():
    with open(os.path.join(REPO_DIR, filename), "wb") as handle:
        handle.write(base64.b64decode(encoded))
sys.path.insert(0, REPO_DIR)
print("Embedded benchmark code ready:", REPO_DIR)


In [ ]:
from pathlib import Path
import csv

manifest_path = Path(MANIFEST)
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")
with manifest_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
required = {"name", "loader", "architecture", "checkpoint_url", "filename"}
if STRICT_44 and len(rows) != EXPECTED_MODELS:
    raise ValueError(f"Expected exactly {EXPECTED_MODELS} models, found {len(rows)}")
if required - set(rows[0]):
    raise ValueError(f"Manifest missing columns: {sorted(required - set(rows[0]))}")
print(f"Validated {len(rows)} manifest rows")
if not STRICT_44:
    print("Test mode: benchmarking the rows currently present in models.csv")

In [ ]:
# Verify the CARDIUM test split before downloading large weights.
from data import CARDIUMDataset
dataset = CARDIUMDataset(DATA_ROOT, split="test", fold=FOLD)
if len(dataset) == 0:
    raise ValueError(
        "No test images found. Expected "
        "cardium_images/cardium_images/fold_N/test/{CHD,Non_CHD}."
    )
print(f"CARDIUM test images: {len(dataset)}")

In [ ]:
# Download/cache weights and evaluate all 44 models.
import sys, subprocess
cmd = [
    sys.executable, os.path.join(REPO_DIR, "kaggle_benchmark.py"),
    "--data-root", DATA_ROOT,
    "--manifest", MANIFEST,
    "--expected-models", str(EXPECTED_MODELS if STRICT_44 else 0),
    "--fold", str(FOLD),
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--cache", CACHE_DIR,
    "--out", RESULTS_CSV,
]
subprocess.run(cmd, check=True)

In [ ]:
# Inspect and rank successful models.
import pandas as pd
results = pd.read_csv(RESULTS_CSV)
display(results.sort_values("roc_auc", ascending=False, na_position="last"))
print(results["status"].value_counts(dropna=False))

In [ ]:
# Package results and downloaded model weights for download from Kaggle.
import shutil
bundle_dir = Path("/kaggle/working/fetal_chd_bundle")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)
shutil.copy2(RESULTS_CSV, bundle_dir / "benchmark_results.csv")
shutil.copy2(MANIFEST, bundle_dir / "models.csv")
shutil.copytree(CACHE_DIR, bundle_dir / "weights")
shutil.make_archive(BUNDLE_ZIP[:-4], "zip", bundle_dir)
print(f"Download this artifact: {BUNDLE_ZIP}")

## Saving to Google Drive

Kaggle cannot directly mount Google Drive. Download `/kaggle/working/fetal_chd_models_and_results.zip` from the notebook Output panel, then upload it to Drive. For a persistent Kaggle copy, save the notebook output as a Kaggle Dataset.